# Can multi-pass residual-GF4 eliminate outlier retention?

**Setting = the same one our outlier-retention results use:** randomized-Hadamard rotation,
**NVFP4 weights** (E2M1 + per-16 E4M3 block scale, plain RTN — *no* Hessian/GPTQ reconstruction),
**GF4 activations**. W4A4 everywhere on the transformer linears; `lm_head`/embeddings stay FP16
(as in the retention setup).

**The test.** The MLP down-projection (`down_proj` in Llama, `fc2` in OPT) is the layer single-pass
FP4 collapses on. We compare, *changing only that layer*:

| condition | down-projection treatment |
|---|---|
| **retention (baseline)** | weights + activations in **FP16** |
| **1-pass** | NVFP4 weight + **1-pass** GF4 activation (plain W4A4, no retention) |
| **N-pass** | NVFP4 weight + **N-pass residual** GF4 activation (multi-pass, no retention) |

Retention is **eliminable** iff some small N makes the N-pass row match the FP16-retention baseline.

> Runtime: 7B fake-quant in FP16 wants an **A100-40GB** Colab runtime. On a smaller GPU use
> `MODEL_ID = "facebook/opt-6.7b"` (non-gated) or a 16-bit device_map. Llama-2 is gated — set your
> HF token in the config cell (Colab: 🔑 *Secrets* → `HF_TOKEN`).

In [ ]:
!pip -q install "transformers>=4.44" datasets accelerate sentencepiece

In [ ]:
import os, math, torch, torch.nn as nn, torch.nn.functional as F

# ---- config ---------------------------------------------------------------
MODEL_ID = "meta-llama/Llama-2-7b-hf"   # or "facebook/opt-6.7b" (non-gated)
SEQLEN   = 2048
NSAMP    = 40          # WikiText-2 windows; raise for tighter PPL
BLOCK    = 16          # micro-scale block (NVFP4/GF4)
CLIP     = 2.5         # GF4 clip ratio (RMS units)
PASSES   = (1, 2, 3, 4)  # down-proj residual passes to sweep vs FP16 retention
DEV      = "cuda"

# HF token for gated Llama-2 (Colab Secrets -> HF_TOKEN, or paste here)
try:
    from google.colab import userdata
    os.environ.setdefault("HF_TOKEN", userdata.get("HF_TOKEN"))
except Exception:
    pass
HF_TOKEN = os.environ.get("HF_TOKEN")

# down-projection layer names that get the switchable treatment
DOWN_SUBSTR = ("down_proj", "fc2")
# layers we keep in FP16 in BOTH conditions (matches the retention setup)
KEEP_FP16   = ("lm_head", "embed_out", "embed_tokens", "embed_positions", "lm_head")

In [ ]:
# ---- codebooks ------------------------------------------------------------
GF4_POS = torch.tensor([0.0, 0.0796082, 0.1737177, 0.2828685,
                        0.3952704, 0.5250730, 0.6961928, 1.0])       # GF4 activation levels
E2M1    = torch.tensor([0.0, 0.5, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0])     # NVFP4 weight levels

def fwht(x):
    """Normalized fast Walsh-Hadamard transform along the last dim (n = power of 2).
    Symmetric + orthonormal, so it is its own inverse."""
    n = x.shape[-1]; orig = x.shape
    x = x.reshape(-1, n).clone()
    h = 1
    while h < n:
        x = x.view(x.shape[0], n // (2 * h), 2, h)
        a = x[:, :, 0, :].clone(); b = x[:, :, 1, :].clone()
        x[:, :, 0, :] = a + b
        x[:, :, 1, :] = a - b
        x = x.reshape(x.shape[0], n)
        h *= 2
    return (x / math.sqrt(n)).reshape(orig)

def rotate(x, D, Kpad):
    """Randomized Hadamard: pad to Kpad, apply random +-1 signs D, then FWHT.
    Same D folded into the weight offline => computational invariance (exact)."""
    K = x.shape[-1]
    if Kpad != K:
        x = F.pad(x, (0, Kpad - K))
    return fwht(x * D)

def to_e4m3(x):
    """Fake-quantize a positive scale tensor to FP8 E4M3 (per NVFP4's block scale)."""
    x = x.float().clamp(min=2.0 ** -9, max=448.0)
    e = torch.floor(torch.log2(x)).clamp(min=-6.0)      # E4M3 min normal exponent
    s = 2.0 ** e
    m = torch.round((x / s) * 8) / 8                     # 3 mantissa bits
    return (m * s).clamp(max=448.0)

def quant_nvfp4(W, block=BLOCK):
    """NVFP4 weight quant (RTN): per-16 E4M3 block scale + nearest E2M1 level. No Hessian."""
    shape = W.shape; K = shape[-1]
    x = W.reshape(-1, block).float()
    absmax = x.abs().amax(-1, keepdim=True).clamp(min=1e-8)
    scale  = to_e4m3(absmax / 6.0)                       # 6.0 = max E2M1 level
    xn     = x / scale
    sign   = xn.sign(); mag = xn.abs().clamp(max=6.0)
    q      = E2M1.to(x.device)
    idx    = (mag.unsqueeze(-1) - q.view(1, 1, -1)).abs().argmin(-1)
    xq     = sign * q[idx] * scale
    return xq.reshape(shape).to(W.dtype)

def gf4(x, block=BLOCK, clip=CLIP):
    """Single-pass GF4 activation quant (RMS*clip block scale, nearest GF4 level)."""
    shape = x.shape; K = shape[-1]
    x2 = x.reshape(-1, K).float()
    pad = (block - K % block) % block
    xp = F.pad(x2, (0, pad)); Kp = xp.shape[1]
    xb = xp.reshape(-1, block)
    rms = xb.pow(2).mean(-1, keepdim=True).sqrt().clamp(min=1e-8)
    scale = rms * clip
    sign = xb.sign(); xn = (xb.abs() / scale).clamp(0, 1)
    L = GF4_POS.to(x.device)
    q = L[(xn.unsqueeze(-1) - L.view(1, 1, -1)).abs().argmin(-1)]
    xh = (sign * scale * q).reshape(x2.shape[0], Kp)[:, :K]
    return xh.reshape(shape).to(x.dtype)

def gf4_npass(x, n, block=BLOCK, clip=CLIP):
    """N-stage residual GF4: q=0; repeat n: q += GF4(x-q). n=1 is plain GF4."""
    xf = x.float(); q = torch.zeros_like(xf)
    for _ in range(n):
        q = q + gf4(xf - q, block, clip)
    return q.to(x.dtype)

In [ ]:
# ---- quantized linear layers ---------------------------------------------
def _sign_vec(Kpad):
    g = torch.Generator().manual_seed(Kpad)             # deterministic per in-dim
    return (torch.randint(0, 2, (Kpad,), generator=g).float() * 2 - 1)

class QuantLinear(nn.Module):
    """Bulk W4A4: rotated NVFP4 weight (offline) + rotated 1-pass GF4 activation (online)."""
    def __init__(self, lin, block=BLOCK, clip=CLIP):
        super().__init__()
        W = lin.weight.data.float().cpu(); out, K = W.shape
        Kpad = 1 << (K - 1).bit_length()
        D = _sign_vec(Kpad)
        Wq = quant_nvfp4(rotate(W, D, Kpad), block)      # fold rotation into weight
        self.register_buffer("D",  D.half())
        self.register_buffer("Wq", Wq.half())
        self.bias = None if lin.bias is None else nn.Parameter(lin.bias.data.clone(), requires_grad=False)
        self.Kpad, self.block, self.clip = Kpad, block, clip
    def forward(self, x):
        xr = rotate(x.float(), self.D.float().to(x.device), self.Kpad)
        y  = gf4(xr, self.block, self.clip).half() @ self.Wq.to(x.device).t()
        if self.bias is not None: y = y + self.bias.to(y.dtype)
        return y.to(x.dtype)

class SwitchDownProj(nn.Module):
    """The down-projection under test. mode='fp16' -> retention; mode='npass' -> FP4 multi-pass.
    Stores BOTH the FP16 weight and the rotated-NVFP4 weight so conditions flip instantly."""
    def __init__(self, lin, block=BLOCK, clip=CLIP):
        super().__init__()
        W = lin.weight.data; out, K = W.shape
        Kpad = 1 << (K - 1).bit_length()
        D = _sign_vec(Kpad)
        Wq = quant_nvfp4(rotate(W.float().cpu(), D, Kpad), block)
        self.register_buffer("D",      D.half())
        self.register_buffer("Wq",     Wq.half())
        self.register_buffer("W_fp16", W.data.clone().half())
        self.bias = None if lin.bias is None else nn.Parameter(lin.bias.data.clone(), requires_grad=False)
        self.Kpad, self.block, self.clip = Kpad, block, clip
        self.mode, self.passes = "fp16", 1               # default: retention
    def forward(self, x):
        if self.mode == "fp16":                          # ---- retention baseline ----
            y = x.to(self.W_fp16.dtype) @ self.W_fp16.to(x.device).t()
        else:                                            # ---- FP4 multi-pass --------
            xr = rotate(x.float(), self.D.float().to(x.device), self.Kpad)
            y  = gf4_npass(xr, self.passes, self.block, self.clip).half() @ self.Wq.to(x.device).t()
        if self.bias is not None: y = y + self.bias.to(y.dtype)
        return y.to(x.dtype)

In [ ]:
# ---- model surgery --------------------------------------------------------
def _set(model, name, mod):
    parent = model.get_submodule(name.rsplit(".", 1)[0]) if "." in name else model
    setattr(parent, name.rsplit(".", 1)[-1], mod)

def wrap_model(model):
    """Replace every transformer nn.Linear with QuantLinear, except the down-projection
    (-> SwitchDownProj) and the FP16-kept head/embeddings."""
    downs = []
    for name, mod in list(model.named_modules()):
        if not isinstance(mod, nn.Linear): continue
        if any(k in name for k in KEEP_FP16): continue
        if any(s in name for s in DOWN_SUBSTR):
            sw = SwitchDownProj(mod).to(DEV); _set(model, name, sw); downs.append(sw)
        else:
            _set(model, name, QuantLinear(mod).to(DEV))
    print(f"  wrapped: {len(downs)} down-projection layers switchable; rest bulk W4A4")
    return downs

def set_down(downs, mode, passes=1):
    for d in downs: d.mode, d.passes = mode, passes

In [ ]:
# ---- load model + WikiText-2 ---------------------------------------------
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset

tok  = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False, token=HF_TOKEN)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16,
                                             token=HF_TOKEN).to(DEV).eval()
test = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")
enc  = tok("\n\n".join(test["text"]), return_tensors="pt").input_ids
downs = wrap_model(model)

@torch.no_grad()
def ppl():
    n = enc.numel() // SEQLEN; nll = cnt = 0
    for i in range(min(n, NSAMP)):
        ids = enc[:, i*SEQLEN:(i+1)*SEQLEN].to(DEV)
        nll += model(ids, labels=ids).loss.float().item() * (SEQLEN - 1)
        cnt += SEQLEN - 1
    return math.exp(nll / cnt)

In [ ]:
# ---- the sweep ------------------------------------------------------------
print(f"== {MODEL_ID}: down-projection FP16-retention vs N-pass FP4 (WikiText-2) ==")
print(f"  {'down-proj treatment':30s} {'PPL':>9s}  {'d vs retention':>15s}")

set_down(downs, "fp16")
base = ppl()
print(f"  {'FP16 retention (baseline)':30s} {base:9.3f}  {'baseline':>15s}")

for p in PASSES:
    set_down(downs, "npass", p)
    v = ppl()
    tag = f"{p}-pass FP4" + (" (plain W4A4)" if p == 1 else " residual")
    print(f"  {tag:30s} {v:9.3f}  {v-base:+15.3f}")
    torch.cuda.empty_cache()

print("\n  Retention is ELIMINABLE if some small N ties the FP16-retention baseline:")
print("  the down-projection then runs on the FP4 array (N passes), no FP16 datapath.")

## Reading the result

- **1-pass FP4** row = plain W4A4 with **no** retention → the collapse retention exists to prevent
  (expect a clearly worse PPL here on the down-projection-sensitive models).
- The smallest **N** whose row returns to the **FP16-retention** baseline is the answer: that many
  residual FP4 passes on `down_proj`/`fc2` **replace** the dedicated FP16 datapath.
- On OPT-125m/1.3B (no rotation, weaker build) N=2 already matched retention. This notebook is the
  faithful-setting 7B confirmation: same Hadamard + NVFP4-RTN + GF4 as the retention results, only the
  down-projection treatment changes.

**Cost note.** N passes touch only the down-projections (~1/L of the MACs) on the *existing* FP4 array;
the alternative is a dedicated FP16 sub-array. If a small N ties retention, multi-pass removes that
extra datapath for a small multiply-count bump instead of added silicon.